# Descarga de ensamblados → Drive

Verifica los ensamblados candidatos contra NCBI y baja los confirmados a
`tesis/70_genomas/`, sin pasar por tu disco.

**Por qué acá:** la sesión de Claude tiene NCBI bloqueado por política, así que
no puede ni verificar ni bajar. Colab sí.

Toda la lógica vive en `scripts/fetch_genomes.sh` — este notebook solo lo
maneja. Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el criterio de
selección de corridas y el de verificación de ensamblados tienen que vivir en
un solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

In [ ]:
import shutil, subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def _actualizar():
    # El clon es un CACHE del repo, no un espacio de trabajo: nada de lo que se
    # escribe durante una corrida vive adentro (el ledger va a Drive). Por eso
    # reset --hard y no pull --ff-only: el pull falla apenas un archivo
    # versionado quede modificado, y fallaba sin hacer ruido, asi que la celda
    # seguia corriendo con el codigo viejo.
    for args in (['fetch', '--depth', '1', 'origin', 'HEAD'],
                 ['reset', '--hard', 'FETCH_HEAD']):
        r = subprocess.run(['git', '-C', str(CLON)] + args,
                           capture_output=True, text=True)
        if r.returncode != 0:
            return False
    return True


def _clonar_de_cero():
    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


def clonar():
    if CLON.exists():
        if _actualizar():
            return 'clon actualizado'
        # Un clon que no se puede actualizar es peor que no tenerlo: la celda
        # seguiria con codigo viejo sin avisar. Se tira y se clona de nuevo.
        shutil.rmtree(CLON)
    return _clonar_de_cero()


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())


In [ ]:
import glob, os, subprocess, shutil

# CADA notebook de Colab corre en su propia VM: lo que instalo otro cuaderno no
# existe aca. Por eso esta celda esta en los cuatro y es idempotente: si las
# herramientas ya estan, no hace nada.
SRA_VER = '3.1.1'
URL = f'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz'


def sh(cmd, t=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)


def _en_path(ruta):
    if ruta and ruta not in os.environ['PATH']:
        os.environ['PATH'] = ruta + ':' + os.environ['PATH']


def instala_sra():
    # ya desempaquetado en esta VM de una corrida anterior de la celda
    c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    if c:
        _en_path(c[0]); 
    if shutil.which('prefetch') and shutil.which('vdb-validate'):
        return 'ya estaba'

    r = sh(f'wget -q -O /tmp/sra.tar.gz "{URL}"')
    if r.returncode == 0 and sh('tar -xzf /tmp/sra.tar.gz -C /opt').returncode == 0:
        c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
        if c:
            _en_path(c[0])
            return f'tarball oficial {SRA_VER}'

    # La version del tarball puede cambiar o desaparecer. apt es mas viejo, pero
    # aca solo se descarga y se valida: nada de esto entra en la tesis.
    if sh('apt-get -qq install -y sra-toolkit').returncode == 0 and shutil.which('prefetch'):
        return 'apt (version distinta del tarball)'

    raise RuntimeError(
        'No pude instalar sra-tools ni por tarball ni por apt. '
        'Revisa la version vigente en https://github.com/ncbi/sra-tools/wiki '
        'y ajusta SRA_VER.')


if not shutil.which('jq'):
    sh('apt-get -qq update'); sh('apt-get -qq install -y jq')
print('sra-tools:', instala_sra())

faltan = [b for b in ('prefetch', 'vdb-validate', 'jq', 'curl', 'git')
          if not shutil.which(b)]
if faltan:
    raise RuntimeError('faltan herramientas: ' + ', '.join(faltan))
print('herramientas OK:', 'prefetch vdb-validate jq curl git')

In [ ]:
GENOMAS = DRIVE / '70_genomas'
GENOMAS.mkdir(parents=True, exist_ok=True)

# Definido una sola vez para que cualquier celda de abajo lo use sin depender de
# haber corrido otra antes.
env_gen = dict(os.environ, GENOMES_DIR=str(GENOMAS))

print('destino:', GENOMAS)


## 1. Qué falta

`candidato` = propuesto pero **no comprobado**. El script se niega a bajar esos
hasta que una persona los verifique. Un ensamblado equivocado no falla
ruidosamente: alinea peor y contamina la anotación.

In [ ]:
!cd /content/tesis && ./scripts/fetch_genomes.sh estado

## 2. Verificar contra NCBI

Para cada organismo pregunta dos cosas: si el accession propuesto existe, y cuál
es el ensamblado de **referencia vigente** de la especie. La segunda importa más
que la primera — un accession puede existir y no ser el que corresponde.

**Leé la salida antes de seguir.**

In [ ]:
!cd /content/tesis && ./scripts/fetch_genomes.sh resolve

## 2b. Ensamblados por cepa

Para cuando la especie **tiene** referencia pero los datos son de **otra cepa**,
y para sacar el accession de un ensamblado del que solo se conoce el nombre.

Dos cosas se resuelven acá:

- **`cloro`**: el ensamblado de IK726 (`GCA_902827195.2`) es la cepa de los datos,
  pero son 70.7 Mb contra una mediana de 55.2 Mb en las otras 20 cepas, y la
  publicación declara ~58 Mb. El `--grep` del accession dice si existe una
  versión `.1` con otro tamaño.
- **Los 3 heredados de R1**: `verificar` los reporta como `SIN RESPALDO` porque
  no tienen accession, así que no hay registro de contra qué se alineó. Del
  nombre del assembly se puede sacar el accession, y con eso pasan al flujo
  normal. **No hace falta `config.sh`.**

Ojo con `phypa`: `Phypa_V3` es nombre de EnsemblGenomes y NCBI no nombra igual —
la trampa Ensembl/NCBI que ya está documentada. Si el `--grep` no encuentra nada,
vaciá el texto de esa fila para listar todo y mirar a ojo.


In [ ]:
# (org, texto a buscar). '' en el texto = lista todo, sin filtro.
CONSULTAS = [
    ('cloro', '902827195'),   # existe una version .1, con otro tamano?
    ('rhirr', 'ASM43914v3'),  # los 3 heredados: sacar el accession del nombre
    ('sclsc', 'ASM14694v1'),
    ('phypa', 'Phypa_V3'),    # OJO: nombre de EnsemblGenomes; NCBI puede no usarlo
]
TAXON = ''   # forzar otro nombre de especie (sinonimos); aplica a todas

for _org, _grep in CONSULTAS:
    cmd = ['./scripts/fetch_genomes.sh', 'cepas', _org]
    if TAXON:
        cmd += ['--taxon', TAXON]
    if _grep:
        cmd += ['--grep', _grep]
    print('$ ' + ' '.join(cmd))
    p = subprocess.Popen(cmd, cwd=CLON, env=env_gen, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for ln in p.stdout:
        print(ln, end='')
    p.wait()
    print()


## 3. Confirmar

Mirá el **veredicto** de cada organismo:

- `COINCIDE` — el candidato es la referencia vigente. Poné el org en
  `CONFIRMADOS`.
- `DIFIERE` — NCBI tiene otra referencia. Gana NCBI: poné el org en `CORREGIR`
  con el accession **y** el nombre del assembly nuevos, y en `CONFIRMADOS`.
- `SIN CANDIDATO` — elegí uno de la lista de cepas y ponelo en `CORREGIR`.
- `SIN RESPUESTA UTIL` — hubo un error de red o NCBI no contestó. **No
  confirmes nada**: volvé a correr la celda 2.

El accession y el nombre del assembly van juntos a propósito. Si se corrige uno
y no el otro, `data/genomas.sha256` termina diciendo que se usó un ensamblado
que no es el que se bajó — y ese fichero es justamente la prueba de contra qué
se alineó.


In [ ]:
CONFIRMADOS = []   # p.ej. ['prupe', 'gadmo'] — solo los que diste por buenos
CORREGIR    = {}   # p.ej. {'galga': ('GCF_000002315.7', 'GRCg6a')}  accession, assembly

spec = CLON / 'data' / 'genomas.tsv'
lineas = spec.read_text().split('\n')
tocadas, vistos = [], set()

for i, ln in enumerate(lineas):
    if ln.startswith('#') or '\t' not in ln:
        continue
    f = ln.split('\t')
    org = f[0]
    if org not in CONFIRMADOS and org not in CORREGIR:
        continue
    vistos.add(org)
    if org in CORREGIR:
        acc, asm = CORREGIR[org]
        f[4], f[3] = acc, asm      # accession y assembly se corrigen juntos
    if org in CONFIRMADOS:
        f[5] = 'verificado'
    lineas[i] = '\t'.join(f)
    tocadas.append(f'  {org:8s} {f[4]:18s} {f[3]:32s} {f[5]}')

# Un typo en CONFIRMADOS no puede pasar en silencio: hoy no haria nada y la
# celda diria que todo salio bien.
faltan = sorted((set(CONFIRMADOS) | set(CORREGIR)) - vistos)
assert not faltan, f'estos no estan en genomas.tsv: {faltan}'

spec.write_text('\n'.join(lineas))
print('\n'.join(tocadas) if tocadas else '(nada que cambiar; no se va a bajar nada)')


## 4. Bajar a Drive

In [ ]:
!cd /content/tesis && GENOMES_DIR=/content/drive/MyDrive/tesis/70_genomas ./scripts/fetch_genomes.sh fetch

## 5. Verificar lo que está bajado

Chequea los **9** organismos contra los archivos, no contra la spec ni el
ledger: que el FASTA exista, que arranque con `>`, y que su `sha256` coincida
con `data/genomas.sha256`. Si coincide, el archivo es idéntico byte a byte al
que se bajó —y como `fetch` lo escribió con gzip, la integridad del stream va
implícita, así que no hace falta leerlo dos veces. Cuando **no** hay entrada en
el ledger, ahí sí corre `gzip -t`, que es lo que detecta un archivo cortado.

Sale con código distinto de cero si algo falla, así que sirve como control antes
de arrancar el alineamiento.

`--rapido` solo mira existencia y la primera línea, sin leer el archivo entero
—que sobre el FUSE de Drive son ~1 GB por genoma.


In [ ]:
ORG    = ''     # '' = los 9, o 'prupe', 'cloro', ...
RAPIDO = False  # True = no verifica checksums

cmd = ['./scripts/fetch_genomes.sh', 'verificar']
if ORG:
    cmd.append(ORG)
if RAPIDO:
    cmd.append('--rapido')

print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env_gen, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
print('exit =', p.wait())


## 6. Cerrar el círculo con git

El clon es efímero: se pierde al cerrar la sesión. Copiá esta salida al repo y
commiteala — **el checksum versionado es lo que deja constancia de qué genoma se
usó**, porque el que queda al lado del FASTA en Drive no prueba nada: quien
reemplace el genoma reemplaza el checksum con él.

In [ ]:
spec = CLON / 'data' / 'genomas.tsv'
led = CLON / 'data' / 'genomas.sha256'
print('--- data/genomas.sha256 ---')
print(led.read_text() if led.exists() else '(vacio: no se bajo nada)')
print('--- data/genomas.tsv ---')
print(spec.read_text())
